# Day 4 v2 — Model 01: DNN + TF-IDF char_wb 100K

**Architecture:** TF-IDF char_wb (2,4) 100K features → sparse mini-batch → PriceDNN (8 ResidualBlocks, hidden=4096)

**Target:** MAE < 80k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

**Key technique:** SparseDataset converts rows on demand — no `.toarray()` on full 269K × 100K matrix.

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch
from sklearn.feature_extraction.text import TfidfVectorizer

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
from pricer_vi_2.deep_neural_network_sparse import SparseDNNRunner

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load Data

In [ ]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

## 2. Setup Model

Fit TF-IDF char_wb (2,4) 100K trên 269K docs. Config nhất quán với Day 3 v2 model 5A (MAE=92.6k).

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(2, 4),
    max_features=100_000,
    sublinear_tf=True,
)

runner = SparseDNNRunner(train, val)
runner.setup(vectorizer, batch_size=256, num_blocks=8, hidden_size=4096)

## 3. Train

Max 10 epochs, early stopping patience=3. Val feedback dùng val[:1000] mỗi epoch.

In [ ]:
history = runner.train(epochs=10, patience=3)

## 4. Training History

In [ ]:
plot_training_history(history, title="DNN + TF-IDF char_wb 100K")

## 5. Save Weights + Val Predictions + Test Predictions

In [ ]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/dnn_tfidf.pth")
print("Saved weights/dnn_tfidf.pth")

Path("val_predictions").mkdir(exist_ok=True)

# Val predictions — full 3926 samples for stacking (Task 7)
print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/dnn_tfidf_val.json", "w") as f:
    json.dump(val_preds, f)

# Test predictions — full 3872 samples for stacking
print("Running test predictions (3872 samples)...")
test_preds = runner.test_predictions(test)
with open("val_predictions/dnn_tfidf_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

## 6. Evaluate on 200 Test Samples

Dùng `pricer_vi_2/evaluator.py` — scatter plot + error trend chart.

In [ ]:
def dnn_tfidf_pricer(item):
    return runner.inference(item)

results = evaluate(dnn_tfidf_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R²: {results['r2']:.1f}%")

## 7. Sanity Check — Load Roundtrip

In [ ]:
# Inference trên trained runner
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred_original:.1f}k VND")
print(f"Error:   {abs(pred_original - sample.price):.1f}k VND")

# Load roundtrip test
runner.load("weights/dnn_tfidf.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch: before={pred_original:.2f} after={pred_loaded:.2f}"
print(f"\nLoad roundtrip PASSED. Diff: {diff:.4f}k")